### **Chapter 9.4: Past I/O History as an Implicit State**

Standard DeePC does not require an explicit state estimator. Instead, a sufficiently long past input-output trajectory identifies the internal state that is relevant for future behavior.

The Mountain Car gives a particularly transparent demonstration. On flat terrain,

$$
x=[p,v]^\top,
$$

and, as everywhere in this chapter, DeePC measures **position only**:

$$
y=p.
$$

The hidden velocity must therefore be encoded by the recent input-position history. This notebook shows why that works and how long the history has to be.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex9_DeePC.deepc_utils import *

### **Part 1: Position-Only Offline Data**

For the discrete double-integrator behavior, one position sample is not enough to determine velocity. Two consecutive position samples (together with the corresponding input history) contain the missing information.

In [ ]:
freq = 20
dt = 1.0 / freq
initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])

env = Env(1, initial_state, target_state, input_lbs=-1.0, input_ubs=1.0)
dynamics = Dynamics(env)

u_data, p_data = collect_deepc_data(
    env,
    dynamics,
    freq=freq,
    n_samples=350,
    excitation_amplitude=0.8,
    initial_state=env.target_state,
    seed=2,
    output_indices=[0],
)

print("u_data:", u_data.shape)
print("position-only y_data:", p_data.shape)

### **Part 2: Fixed-Input Prediction with Different History Lengths**

To isolate the role of the initialization trajectory from the control objective, we first perform a pure behavioral prediction experiment.

A test trajectory is generated from a state with nonzero velocity. We then reveal only position measurements and ask the Hankel predictor to continue the trajectory under a known future input sequence.

In [ ]:
def rollout(dynamics, x0, u_sequence, dt):
    x = np.asarray(x0, dtype=float).copy()
    states = [x.copy()]
    for u in np.asarray(u_sequence):
        x = dynamics.one_step_forward(x, np.asarray(u).reshape(-1), dt)
        states.append(x.copy())
    return np.asarray(states)

rng = np.random.default_rng(20)
N_pred = 20
n_test = 40
u_test = rng.uniform(-0.45, 0.45, size=(n_test, 1))
x_test = rollout(dynamics, np.array([-0.25, 0.32]), u_test, dt)
p_test = x_test[:, [0]]

# Use the same current time for all history lengths.
k0 = 6
results = {}
for T_ini in [1, 2, 4]:
    u_ini = u_test[k0-T_ini:k0]
    y_ini = p_test[k0-T_ini:k0]
    u_future = u_test[k0:k0+N_pred]

    y_pred, g, eq_residual, null_gain = predict_future_from_history(
        u_data,
        p_data,
        u_ini,
        y_ini,
        u_future,
        output_offset=np.array([env.target_position]),
    )
    results[T_ini] = (y_pred, eq_residual, null_gain)

    print(
        f"T_ini={T_ini}: equality residual={eq_residual:.2e}, "
        f"future-output null gain={null_gain:.2e}"
    )

In [ ]:
t = np.arange(N_pred) / freq
p_true = p_test[k0:k0+N_pred, 0]

plt.figure(figsize=(9, 4))
plt.plot(t, p_true, linewidth=2, label="true future position")
for T_ini, (p_pred, _, _) in results.items():
    plt.plot(t, p_pred[:, 0], "--", label=f"DeePC prediction, T_ini={T_ini}")
plt.xlabel("Prediction time (s)")
plt.ylabel("position")
plt.title("Position-only prediction: history resolves hidden velocity")
plt.legend()
plt.tight_layout()
plt.show()

The `future-output null gain` measures whether there are directions in the nullspace of the past-I/O and future-input constraints that can still change the future output:

$$
\max_{z\in\ker A,\,\|z\|=1}\|Y_f z\|.
$$

- a nonzero value means the same supplied information admits different future outputs;
- a value near zero means the future output is uniquely determined by the supplied history and future input.

This gives a direct numerical view of why $T_{\mathrm{ini}}$ must be at least the system lag/observability index.

### **Part 3: Position-Only DeePC Control**

The controller receives the full simulator state, but `output_indices=[0]` ensures that **only the position is used by DeePC** (exactly as in Chapters 9.1-9.3). Velocity is never inserted into the Hankel constraints or cost. Here we use the shortest history that works, $T_{\mathrm{ini}} = 2$, the lag of the double integrator.

For this textbook output-feedback demonstration, `enforce_current_output=False`: the first future output is inferred from the past I/O history instead of being separately fixed from the current state.

In [ ]:
Q_y = np.array([[1.0]])
R = np.array([[0.1]])
Qf_y = Q_y
N = 20
T_ini = 2

controller_position_only = DeePCController(
    env,
    dynamics,
    u_data,
    p_data,
    Q_y,
    R,
    Qf_y,
    freq,
    N,
    T_ini=T_ini,
    lambda_g=1e-6,
    output_indices=[0],
    enforce_current_output=False,
    history_initialization='equilibrium',
    name='DeePC_position_only',
    verbose=False,
)

def run_position_only_closed_loop(controller, dynamics, env, freq, t_terminal):
    x = env.init_state.copy()
    states = [x.copy()]
    inputs = []
    predictions = []

    for k in range(int(freq * t_terminal)):
        u, y_pred, _ = controller.compute_action(x, k)
        predictions.append(y_pred.copy())
        x = dynamics.one_step_forward(x, u, 1.0/freq)
        states.append(x.copy())
        inputs.append(np.asarray(u).reshape(-1))

    return np.asarray(states), np.asarray(inputs), predictions

x_pos, u_pos, p_predictions = run_position_only_closed_loop(
    controller_position_only, dynamics, env, freq, t_terminal=8
)

In [ ]:
t_x = np.arange(len(x_pos)) / freq
t_u = np.arange(len(u_pos)) / freq

fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
ax[0].plot(t_x, x_pos[:, 0])
ax[0].axhline(env.target_position, linestyle=":", label="target")
ax[0].set_ylabel("measured position")
ax[0].legend()

ax[1].plot(t_x, x_pos[:, 1])
ax[1].set_ylabel("true velocity (not given to DeePC)")

ax[2].plot(t_u, u_pos[:, 0])
ax[2].set_ylabel("input")
ax[2].set_xlabel("Time (s)")

fig.suptitle("Closed-loop DeePC with position-only output")
plt.tight_layout()
plt.show()

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: In DeePC, past I/O can play the role of an implicit state**

With partial measurements, the current output alone need not determine the future. Once the initialization horizon is long enough, the past trajectory contains the information that a model-based controller would normally obtain from an explicit state or observer.
</blockquote>